# Data Cleaning for Project

All data was collected from the `data.census.gov` website. Below is the code used for cleaning and formatting the raw datasets into a malleable form for further analysis.

In [1]:
import pandas as pd
from pathlib import Path
import openpyxl

DIR = Path("__file__").resolve().parent
DATA_DIR = DIR / "raw_data"

CLEAN_DATA_DIR = DIR / "clean_data"

STANDARDIZED_DATA = DIR / "strd_data"

CLUSTER_DIR = DIR / "clustering"

### Cleaning File 1: **total_population.csv**

The data was sourced from: https://data.census.gov/table/ACSST5Y2020.S0101?t=Populations+and+People&g=040XX00US51$8600000&y=2020&tp=true

In [32]:
def clean_file1():
    
    file_path = DATA_DIR / "total_population.csv"
    data_types = {
        "Label (Grouping)": str,
        "Total": int
    }
    df = pd.read_csv(file_path).dropna(subset=['Total'])
    df.columns = ['ZIP', 'Total Population']
    
    df['Total Population'] = df['Total Population'].str.replace(',', '').astype(int)
    df['ZIP'] = df['ZIP'].str.replace('ZCTA5 ', '').astype(int)

    df.to_csv(CLEAN_DATA_DIR / "total_pop.csv", index=False) 

clean_file1()

### Cleaning the File 2 **population_demographics.csv**

The data was sourced from: https://data.census.gov/table/DECENNIALDHC2020.P1?t=Populations+and+People&g=040XX00US51$8600000&y=2020&tp=true

In [6]:
def parse_column_names(names: list[str]) -> list[str]:

    cleaned_names = ["ZIP"]
    for n in names[1:]:
        chunks = n.split('!!')
        if len(chunks) == 3:
            column_name = f"{chunks[1]} Population"
            cleaned_names.append(column_name)

        elif len(chunks) == 5:
            column_name = f"{chunks[1]} {chunks[-1]}"
            cleaned_names.append(column_name)

        else:
            print("uhoh")

    return cleaned_names
def clean_file2():

    file_path = DATA_DIR / "population_demographics.csv"
    df = pd.read_csv(file_path)
    print(df.shape)
    df = df.drop(df.filter(like='Margin').columns, axis=1)
    df = df.drop(df.filter(like='Percent').columns, axis=1)
    df = df.drop(df.filter(like='PERCENT').columns, axis=1)
    df = df.drop(df.filter(like='SUMMARY').columns, axis=1)
    df = df.drop(df.filter(like='SELECTED').columns, axis=1)

    column_names = df.columns.to_list()
    clean_column_names = parse_column_names(column_names)
    df.columns = clean_column_names

    df['ZIP'] = df['ZIP'].str.replace('ZCTA5 ', '').astype(str)


    df.to_csv(CLEAN_DATA_DIR / "pop_demo.csv", index=False)

clean_file2()

(897, 457)


### Cleaning the File 3: **employement_and_education.csv**

This data was sourced from: https://data.census.gov/table/ACSDT5Y2020.B23006?t=Education:Employment&g=040XX00US51$8600000&y=2020&tp=true

It covers the status of all adults *aged 25-64*

In [ ]:
def parse_column_names(names: list[str]) -> list[str]:

    cleaned_names = ["ZIP", "Total in Working Age"]

    for n in names[2:]:
        chunks = n.split('!!')[2:]
        cleaned_names.append('~'.join(chunks).replace(':', ''))
    
    return cleaned_names

def clean_file3():

    file_path = DATA_DIR / "employement_and_education.csv"
    df = pd.read_csv(file_path)
    print(df.shape)
    df = df.drop(df.filter(like='Margin').columns, axis=1)

    column_names = df.columns.to_list()
    clean_column_names = parse_column_names(column_names)
    df.columns = clean_column_names

    df['ZIP'] = df['ZIP'].str.replace('ZCTA5 ', '').astype(str)
    df.set_index('ZIP', inplace=True)

    df.to_csv(CLEAN_DATA_DIR / "emp_and_edu.csv", index=True)

clean_file3()

(897, 59)


### Cleaning File 4: **occupation.csv**

The data was sourced from: https://data.census.gov/table/ACSST5Y2020.S2406?t=Class+of+Worker&g=040XX00US51$8600000&y=2020&tp=true

Segments the Employed Civillian Population (16+) by role type

In [13]:
def parse_column_names(names: list[str]) -> list[str]:

    cleaned_names = ["ZIP", "Total Employed Civillians Above 16"]

    for n in names[2:]:
        chunks = n.split('!!')
        print(chunks[-1])
        cleaned_names.append(chunks[-1])
    
    return cleaned_names

def clean_file4():
    
    file_path = DATA_DIR / 'occupation.csv'
    df = pd.read_csv(file_path)
    
    df = df.drop(df.filter(like='Margin').columns, axis=1)
    df = df.drop(df.filter(like='Percent').columns, axis=1)
    df = df.drop(df.filter(like='PERCENT').columns, axis=1)



    df.rename(columns={df.columns[0]: 'ZIP'}, inplace=True)

    df['ZIP'] = df['ZIP'].str.replace('ZCTA5 ', '').astype(str)

    df = df.filter(regex='ZIP|Total', axis=1).dropna() 

    df.columns = parse_column_names(df.columns.to_list())
    df.set_index('ZIP', inplace=True)

    df.to_csv(CLEAN_DATA_DIR / 'occupation.csv', index=True)

clean_file4()

Management, business, science, and arts occupations
Service occupations
Sales and office occupations
Natural resources, construction, and maintenance occupations
Production, transportation, and material moving occupations


## Standardizing the information between files

The data from **American Community Survey** (population_demographics.csv) and **Deccenial Census** (total_population.csv) contain conflicting information on ZIP code counts and population size for the *Commonwealth of Virginia*. We're going to operate under the assumption that the **Deccenial Census** is a more trustworthy source for total population size than **American Community Survey** and use only the ZIP codes listed from both data sources.

In [15]:
def remove_excess_ZIP_codes():
    
    df_deccenial = pd.read_csv(CLEAN_DATA_DIR / "total_pop.csv", index_col='ZIP')
    df_acs = pd.read_csv(CLEAN_DATA_DIR / "pop_demo.csv", index_col='ZIP')

    common_zips = df_deccenial.index.intersection(df_acs.index)

    merged_data = df_acs.loc[common_zips].copy()
    merged_data['Total Population'] = df_deccenial.loc[common_zips, 'Total Population']

    merged_data.to_csv(STANDARDIZED_DATA / 'MERGED_POPULATION_DEMOGRAPHIC_DATA.csv')

    df_emp_edu = pd.read_csv(CLEAN_DATA_DIR / 'emp_and_edu.csv', index_col='ZIP')
    standardized_emp_edu_df = df_emp_edu.loc[common_zips].copy()

    standardized_emp_edu_df.to_csv(STANDARDIZED_DATA / 'CLEANED_EMPLOYEMENT_EDUCATION_DATA.csv', index=True)

    df_income = pd.read_csv(CLEAN_DATA_DIR / 'occupation.csv', index_col='ZIP')
    standardized_income = df_income.loc[common_zips].copy()

    standardized_income.to_csv(STANDARDIZED_DATA / 'OCCUPATION_DATA.csv', index=True)


remove_excess_ZIP_codes()

In [11]:
def refine_edu_df(file_path: Path) -> pd.DataFrame:
    
    df_education = pd.read_csv(file_path, index_col='ZIP')
    df_education = df_education.drop(df_education.filter(like='~').columns, axis=1)

    return df_education

def create_composite():

    edu_df = refine_edu_df(STANDARDIZED_DATA / 'CLEANED_EMPLOYEMENT_EDUCATION_DATA.csv')

    gender_pop_df = pd.read_csv(CLUSTER_DIR / "demographics" / "gender_log_ratio.csv", index_col='ZIP')

    occupation_df = pd.read_csv(STANDARDIZED_DATA / 'OCCUPATION_DATA.csv', index_col='ZIP')

    total_pop_df = pd.read_csv(STANDARDIZED_DATA / 'MERGED_POPULATION_DEMOGRAPHIC_DATA.csv', usecols=['ZIP', 'Total Population'])
    total_pop_df.set_index('ZIP')
    combined = edu_df.join([total_pop_df, gender_pop_df, occupation_df])

    df_info = pd.DataFrame(
        {
            "Column": combined.columns,
            "DataType": combined.dtypes.values
        }
    )
    # df_info.to_csv('data_legend.csv', index=False)
    combined.to_excel(STANDARDIZED_DATA / 'ALL_COMPUTABLE_DATA.xlsx', index=True)

refine_edu_df(STANDARDIZED_DATA / 'CLEANED_EMPLOYEMENT_EDUCATION_DATA.csv').to_csv(STANDARDIZED_DATA / 'SIMPLIFIED_EDU.csv')